# [yoctogpt](https://github.com/Infatoshi/yoctogpt)
> Generatively Pre-train a Transformer in 100 lines of python

You're about to watch a computer go from **pure gibberish** to **recognizable English**, learning to complete "Peter Piper picked a peck of pickled peppers" from scratch.

**Before training:**
```
Peter Piper 'sa'kIIeWkpp ep.sci.,cpk e osseIh
```

**After training:**
```
Peter Piper picked a peck of pickled peppers
```

The entire "brain" that learns this is just **~4,000 numbers** that start random and get slowly adjusted. That's it. No magic.

**What you need:** basic Python (lists, loops, classes). No machine learning background. No math beyond arithmetic. There are no dependencies and nothing to install: just run the cells top to bottom.

This is [micrograd](https://github.com/karpathy/micrograd) + [microgpt](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95) (inspired by [nanogpt](https://github.com/karpathy/nanoGPT)), expanded with explanations. The compressed version of everything here, the same model in under 100 lines, lives in [`v1.py`](https://github.com/Infatoshi/yoctogpt/blob/master/v1.py). Learn here first; read the 100-liner after, as a victory lap.


## How This Works (No Math Version)

Here's the entire idea in plain English:

1. **Start with random guesses** — The computer begins with ~4,000 random numbers. These numbers are its "brain."

2. **Make a prediction** — Given some letters like `"Peter P"`, it guesses what comes next. At first, it's terrible.

3. **Check the answer** — We compare its guess to the actual next letter (`"i"` in "Piper").

4. **Measure the mistake** — We calculate how wrong it was. This is called the **loss**.

5. **Adjust the numbers** — We figure out which of those 4,000 numbers to nudge up or down to make the guess slightly better. This is called **backpropagation**.

6. **Repeat 1,000 times** — Each time, the guesses get a tiny bit better.

7. **Watch it learn** — Gibberish → fragments → recognizable phrases.

That's it. Everything below is just the Python code to make this happen.

### The Big Picture

Here's the entire model at a glance — from input character to output prediction:

![Full GPT Model](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/06-full-gpt.png)

---
## 1 · Data & Tokenization

We use a tiny dataset — the "Peter Piper" tongue twister — so training finishes in minutes on a CPU. The tokenizer is character-level: every unique character gets an integer ID. We then split the encoded sequence 90/10 into training and validation sets.

**Why characters?** Real LLMs like ChatGPT use "subword tokens" (chunks like `"ing"` or `"the"`), but characters are simpler to understand. Same concept, just smaller pieces.

**Why split the data?** We hide 10% of the text during training. If the model can predict those unseen characters, it actually *learned the pattern* rather than just memorizing.
**Why the word "token"?** Like an arcade token: a small stand-in. We swap each piece of text for a numbered counter that the machine can do arithmetic on. The text and the numbers are interchangeable; nothing is lost.


In [1]:
import math
import random

random.seed(67)

txt = "Peter Piper picked a peck of pickled peppers.\nA peck of pickled peppers Peter Piper picked.\nIf Peter Piper picked a peck of pickled peppers,\nWhere's the peck of pickled peppers Peter Piper picked?\n"


In [2]:
len(txt)


196

In [3]:
characters = sorted(set(txt))
vocab_size = len(characters)

print(f"vocab size: {vocab_size}")
print(f"characters: {characters}")

vocab size: 23
characters: [' ', "'", ',', '.', '?', 'A', 'I', 'P', 'a', 'c', 'd', 'e', 'f', 'h', 'i', 'k', 'l', 'o', 'p', 'r', 's', 't', 'w']


In [4]:
token_ids = [characters.index(char) for char in txt]

print(f"encoded length: {len(token_ids)}")
print(f"first 50 tokens: {token_ids[:50]}")

encoded length: 196
first 50 tokens: [7, 11, 21, 11, 19, 0, 7, 14, 18, 11, 19, 0, 18, 14, 9, 15, 11, 10, 0, 8, 0, 18, 11, 9, 15, 0, 17, 12, 0, 18, 14, 9, 15, 16, 11, 10, 0, 18, 11, 18, 18, 11, 19, 20, 3, 0, 5, 0, 18, 11]


In [5]:
split_point = int(len(token_ids) * 0.9)
train_data = token_ids[:split_point]  # first 90% — model learns from this
val_data = token_ids[split_point:]    # last 10% — we test on this (model never sees it during training)

print(f"train: {len(train_data)} tokens, val: {len(val_data)} tokens")

train: 176 tokens, val: 20 tokens


In [6]:
# TRY IT YOURSELF

# What number is the letter 'P'?
# print(f"'P' is token number: {characters.index('P')}")

# Encode your own text:
# my_text = "Pepper"
# my_ids = [characters.index(c) for c in my_text]
# print(f"'{my_text}' encodes to: {my_ids}")

# Decode it back:
# decoded = ''.join([characters[i] for i in my_ids])
# print(f"Decoded back: '{decoded}'")

### Hyperparameters (settings we choose)

These are the "knobs" we set before training:

- `ctx = 8` — The model can only "see" the last 8 characters when predicting. (frontier models like Claude Opus or GPT-5.5 see hundreds of thousands)
- `d = 16` — Each token becomes a 16-number vector. More numbers = more "meaning" capacity.
- `nl = 1` — Just one layer of attention. Real models have 32-96 layers.
- `lr = 0.01` — Learning rate. How big of a step to take each update.

In [7]:
context_length = 8
embed_dim = 16
num_layers = 1
learning_rate = 0.01

print(f"context={context_length}, embed_dim={embed_dim}, layers={num_layers}, lr={learning_rate}")

context=8, embed_dim=16, layers=1, lr=0.01


---
## 2 · Autograd Engine

**This is the hardest section conceptually. Go slow.**

### The Core Question: "If I tweak this number, how does the output change?"

Imagine you have a dial labeled `x` and a screen showing `y = x²`.
- When `x = 3`, the screen shows `y = 9`
- Nudge the dial to `x = 3.001`, the screen shows `y ≈ 9.006`
- The screen changed ~6× as fast as the dial

That ratio (≈6) is the **gradient**. It answers: *"Which direction should I turn the dial to make the screen's number go down?"*

**Why does this matter?** We have 4,000 dials (parameters). We need to know which way to nudge *all* of them to make the loss decrease. Computing this by hand would be insane — so we build a system that tracks it automatically.

### What class `V` does

The `V` class is a "smart number" that remembers how it was computed:
- `.data` = the actual number
- `.grad` = how much the final result changes when you tweak this number
- When you do `a + b` or `a * b`, the result remembers its "parents"
- Calling `.backward()` walks through every operation in reverse, filling in all gradients

This is exactly what PyTorch does under the hood. We're just building a tiny version. ([micrograd](https://github.com/karpathy/micrograd) is the inspiration)

### Visualizing the Computation Graph

Every operation creates a node in the graph. Gradients flow backward through it:

![Computation Graph](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/01-computation-graph.png)

In [8]:
# Let's prove the gradient idea manually BEFORE using class V:
# If y = x², then dy/dx = 2x. At x=3, that's 6.

x = 3.0
y = x ** 2  # y = 9

# Nudge x by a tiny amount
x_nudged = 3.001
y_nudged = x_nudged ** 2  # 9.006001

# How much did y change per unit change in x?
dy = y_nudged - y       # 0.006001
dx = 0.001
slope = dy / dx          # ≈ 6.0

print(f"Manual gradient: {slope:.4f}")
print(f"Calculus says:   {2 * x:.4f}")  # derivative of x² is 2x

Manual gradient: 6.0010
Calculus says:   6.0000


In [10]:
class Value:
    """A 'smart number' that tracks how it was computed, enabling automatic gradient calculation."""

    def __init__(self, data, children=(), local_gradients=()):
        self.data = data                    # the actual number
        self.grad = 0.0                     # gradient: how much does the final output change if we tweak this?
        self.children = children            # what Values were used to compute this?
        self.local_grads = local_gradients  # how does this Value change w.r.t. each child?

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # d(a+b)/da = 1, d(a+b)/db = 1
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # d(a*b)/da = b, d(a*b)/db = a
        return Value(self.data * other.data, (self, other), (other.data, self.data))

    def __pow__(self, n):
        # d(x^n)/dx = n * x^(n-1)
        return Value(self.data**n, (self,), (n * self.data ** (n - 1),))

    def __sub__(self, other):
        return self + (other * -1)

    def log(self):
        # d(log(x))/dx = 1/x
        return Value(math.log(self.data), (self,), (1 / self.data,))

    def exp(self):
        # d(e^x)/dx = e^x
        result = math.exp(self.data)
        return Value(result, (self,), (result,))

    def relu(self):
        # d(relu(x))/dx = 1 if x > 0 else 0
        return Value(max(0, self.data), (self,), (float(self.data > 0),))

    def backward(self):
        """Compute gradients for all Values that contributed to this one."""
        # Build topological order (children before parents)
        topo_order, visited = [], set()
        def build_order(node):
            if node not in visited:
                visited.add(node)
                for child in node.children:
                    build_order(child)
                topo_order.append(node)
        build_order(self)

        # Backpropagate: walk backwards, accumulating gradients
        self.grad = 1.0  # d(output)/d(output) = 1
        for node in reversed(topo_order):
            for child, local_grad in zip(node.children, node.local_grads):
                child.grad += local_grad * node.grad  # chain rule!


In [11]:
# quick sanity check: d/dx (x^2) at x=3 should be 6
x = Value(3.0)
y = x ** 2
y.backward()
print(f"x={x.data}, y=x^2={y.data}, dy/dx={x.grad}")

x=3.0, y=x^2=9.0, dy/dx=6.0


**What just happened?** `x.grad = 6.0`: same answer as the manual nudge, but automatic, for any computation. The entire GPT forward pass is just adds and multiplies; `backward()` handles all of it.

Now the picture that matters: one trainable neuron, and where the blame goes when `backward()` runs.

![One Neuron: where the blame goes](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/10-one-neuron.png)

Run it yourself:


In [ ]:
# one tiny neuron, backward by hand: y = x * w, and we wanted 10
x = Value(3.0)   # pretend this is the previous layer's output
w = Value(2.0)   # this is ours to train
y = x * w        # y = 6.0

loss = (y - 10) ** 2   # squared error: (6 - 10)^2 = 16
loss.backward()

print(f"y's blame (dLoss/dy): 2*(6-10) = -8   ('too small: increase me')")
print(f"w.grad = {w.grad}   (= x.data * -8 = 3 * -8)")
print(f"x.grad = {x.grad}   (= w.data * -8 = 2 * -8 -> the message sent leftward)")

# take one training step on w and watch y move toward the target
w.data -= 0.01 * w.grad
print(f"after one nudge: w = {w.data:.2f}, y = x*w = {3.0 * w.data:.2f}  (was 6.0, heading to 10)")


### Chain Rule in Action

When we have multiple operations, gradients multiply through the chain:

![Chain Rule](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/02-chain-rule.png)

---
## 3 · Model Parameters

**The "brain" is just a bunch of numbers organized into tables.**

We create several lookup tables and weight matrices, all filled with small random numbers:

| Name | Shape | Purpose |
|------|-------|---------|
| `token_embed` | 24 × 16 | **Token embeddings**: each of our 24 characters gets a 16-number "meaning vector" |
| `position_embed` | 8 × 16 | **Position embeddings**: each position (0-7) gets a 16-number "location vector" |
| `output_proj` | 24 × 16 | **Output projection**: converts the internal representation back into character scores |
| `query, key, value, attn_out` | 16 × 16 each | **Attention projections**: used by tokens to "look at" each other |
| `ff_up, ff_down` | 64 × 16, 16 × 64 | **Feed-forward network**: a small "thinking" layer |

**Total: ~4,000 numbers.** That's it. Frontier models like Claude Opus or GPT-5.5 have trillions. Same idea, different scale.


### How the brain is stored

In section 2, one parameter was a single `Value`: a number that remembers its gradient. Everything bigger is just those, arranged:

![Scalars to Matrices](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/09-scalars-to-matrices.png)

- a **vector** is a plain Python list of 16 Values (one embedding)
- a **matrix** is a list of those lists (24 rows)
- the **whole model** is a dict of named matrices: ~4,000 Values total

That's all `make_matrix` builds below. When PyTorch says "tensor," it means these same tables, packed tightly in memory and processed thousands-at-a-time on a GPU. Same objects, industrial packaging.


### How Embeddings Work

Each character becomes a vector of learnable numbers:

![Token Embedding](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/03-token-embedding.png)
**Why the word "embedding"?** Characters are separate symbols with no notion of "close" or "far": 'P' isn't *near* 'p' in any built-in way. So we *embed* them into a continuous space: each character becomes a point in 16-dimensional space, and now distance and direction exist. Training moves these points around so that characters used similarly drift close together.

**Why a *position* embedding too?** Attention (section 5) treats its inputs as a bag: it has no built-in sense of order, so "Pa" and "aP" would look identical. The fix: every slot (0 through 7) also gets a learned 16-number vector that means "I am in slot 3." We add it to the token's vector, so one summed vector says both *what* the character is and *where* it sits.


In [12]:
def make_matrix(rows, cols):
    """Create a rows × cols matrix of Value scalars, initialized with small random numbers."""
    # Small random numbers (mean=0, std=0.08) help training start smoothly
    return [[Value(random.gauss(0, 0.08)) for _ in range(cols)] for _ in range(rows)]

In [13]:
weights = {
    "token_embed":    make_matrix(vocab_size, embed_dim),      # 24 × 16: each character gets a 16-number vector
    "position_embed": make_matrix(context_length, embed_dim),  # 8 × 16: each position gets a 16-number vector
    "output_proj":    make_matrix(vocab_size, embed_dim),      # 24 × 16: convert back to character probabilities
}

for layer_idx in range(num_layers):
    weights[f"layer{layer_idx}.query"]    = make_matrix(embed_dim, embed_dim)  # 16 × 16
    weights[f"layer{layer_idx}.key"]      = make_matrix(embed_dim, embed_dim)  # 16 × 16
    weights[f"layer{layer_idx}.value"]    = make_matrix(embed_dim, embed_dim)  # 16 × 16
    weights[f"layer{layer_idx}.attn_out"] = make_matrix(embed_dim, embed_dim)  # 16 × 16

    weights[f"layer{layer_idx}.ff_up"]   = make_matrix(embed_dim * 4, embed_dim)  # 64 × 16
    weights[f"layer{layer_idx}.ff_down"] = make_matrix(embed_dim, embed_dim * 4)  # 16 × 64

all_parameters = [param for matrix in weights.values() for row in matrix for param in row]

print(f"total parameters: {len(all_parameters):,}")

total parameters: 3,936


---
## 4 · Neural Net Primitives

Three helper functions used everywhere in the model. The names sound fancy; the math is arithmetic.

### `linear(x, w)`: matrix × vector, a.k.a. "projection"

This is THE core operation of neural networks (it's what GPUs spend ~99% of their time doing). Walk through it once with real numbers and the word "projection" stops being scary:

![Matrix times Vector](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/08-matrix-vector.png)

For every **row** of `w`, compute a dot product with `x`. Each row asks one question: *"how much does x point in my direction?"* The answers, stacked, are the output vector.

So why call it a **projection**? Because it maps the input into a different space: 16 numbers in, 24 numbers out for `output_proj` (one score per character). The input isn't changed; it is *viewed from* 24 learned angles. When you see `query`, `key`, `value`, `attn_out`, `ff_up`, `ff_down`, `output_proj`: all of them are exactly this one operation with different learned matrices.

### `normalize(x)`: keep the numbers tame

The actual math (this is "RMS normalization"):

1. mean square: `ms = (x₀² + x₁² + ... + x₁₅²) / 16`
2. divide every element by `√ms`

Example: `normalize([3, -4])` → ms = (9 + 16) / 2 = 12.5, √ms ≈ 3.54 → `[0.85, -1.13]`. The direction of the vector is preserved; its typical size is forced to ~1.

**Why bother?** x flows through layer after layer, multiplied by weights at every step. Numbers slightly too big grow exponentially (explode); slightly too small decay to nothing (vanish). Either way the gradients become useless and learning dies. Normalizing between operations resets the scale every time. (Production models use LayerNorm; same purpose, slightly different formula.)

### `relu` and why we need a **nonlinearity**

`relu(x) = max(0, x)`: negatives become 0, positives pass through. A bent wire.

Here's the deep reason it must exist: a stack of linear layers *without* it collapses into one linear layer. Two matrix multiplies in a row are always equal to some single matrix multiply, so a 96-layer network would compute nothing a 1-layer network couldn't. Depth would be an illusion. Run the proof two cells down. The little bend in relu is what breaks the collapse and lets depth build genuinely new shapes.

### `softmax(xs)`: scores → probabilities

Exponentiate every score, divide by the sum: now everything is positive and sums to 1. Bigger scores get disproportionately bigger shares. (We subtract the max first so `exp()` can't overflow; the subtraction cancels out in the division.)


In [14]:
def linear(input_vector, weight_matrix):
    output = []
    for row in weight_matrix:
        dot_product = Value(0)
        for w, x in zip(row, input_vector):
            dot_product = dot_product + (w * x)
        output.append(dot_product)
    return output

In [15]:
def normalize(vector):
    sum_sq = Value(0)
    for x in vector:
        sum_sq = sum_sq + (x * x)

    mean_squared = sum_sq * (1 / len(vector))
    scale = (mean_squared + 1e-5) ** -0.5

    result = []
    for x in vector:
        result.append(x * scale)
    return result

In [16]:
def softmax(scores):
    max_val = max(v.data for v in scores)

    exp_scores = []
    for x in scores:
        exp_scores.append((x - max_val).exp())

    total = Value(0)
    for x in exp_scores:
        total = total + x

    if total.data == 0:
        return []

    probs = []
    for x in exp_scores:
        probs.append(x * (total ** -1))
    return probs

In [17]:
# quick test: softmax of [1.0, 2.0, 3.0] should sum to 1.0
test_result = softmax([Value(1), Value(2), Value(3)])
print(f"softmax([1,2,3]) = {[round(v.data, 4) for v in test_result]}")
print(f"sum = {sum(v.data for v in test_result):.6f}")

softmax([1,2,3]) = [0.09, 0.2447, 0.6652]
sum = 1.000000


In [ ]:
# PROOF: two linear layers with no relu collapse into one linear layer
W1 = [[0.5, -1.0], [2.0, 1.0]]
W2 = [[1.0, 1.0], [-1.0, 0.5]]

def matvec(W, v):
    return [sum(w * vi for w, vi in zip(row, v)) for row in W]

v = [3.0, -2.0]
two_layers = matvec(W2, matvec(W1, v))

# combine W2 and W1 into a single matrix, then apply it once
W_combined = [[sum(W2[i][k] * W1[k][j] for k in range(2)) for j in range(2)] for i in range(2)]
one_layer = matvec(W_combined, v)

print(f"two linear layers: {two_layers}")
print(f"one combined layer: {one_layer}")
# identical. depth without a nonlinearity is fake depth.
# relu's bend (max(0, x)) cannot be written as any matrix, so it breaks this collapse.


---
## 5 · GPT Forward Pass

This is the heart of the model. Given the tokens so far, it produces a score for every possible next character. (High score = "probably this one next.")

### The flow (simplified):

```
Token 'P' at position 0
    ↓
Look up embedding for 'P' (16 numbers)
Add position embedding for "slot 0" (16 numbers)  
    ↓
ATTENTION: "Look at all previous tokens, decide what's relevant"
    - Query: "What am I looking for?"
    - Keys:  "What does each previous token contain?"
    - Values: "What information to actually pass forward?"
    ↓
FEEDFORWARD: "Think about what I found"
    ↓
Output: 24 raw scores, one per possible next character
    ↓
(later: softmax turns scores into probabilities)
    ↓
Prediction: "i" has highest probability (for "Pi" → "Piper")
```

### Why attention matters
Without attention, token 5 has no idea what tokens 0-4 said. With attention, it can "look back" and decide: *"Token 2 was a 'P' — that's relevant to my prediction."*
### Why is it called "attention"?

Mechanically, *paying attention to something* = *giving it a bigger share of a weighted average*. Each token blends the `Value` vectors of all earlier tokens into one summary, and the blend weights come from the Q·K relevance scores. "Attending to token 2 with weight 0.45" literally means: 45% of my new representation is token 2's contribution. That's the whole metaphor: the model spends a fixed budget of focus (the weights sum to 1) across the past, exactly the way your attention splits across a noisy room.


### Attention: "Which tokens matter?"

The model learns to focus on relevant previous tokens. Note the grayed-out future token: a token may only look backward, never ahead. That rule is what makes this a *causal* (autoregressive) model:

![Attention Mechanism](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/04-attention.png)

### The Transformer Block

One layer of the model — attention + feedforward with skip connections:

![GPT Block](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/05-gpt-block.png)

In [18]:
def gpt_forward(token_ids, return_attention=False):
    """Process a sequence of tokens and return logits for each position."""
    seq_len = len(token_ids)
    keys = [[] for _ in range(num_layers)]
    values = [[] for _ in range(num_layers)]
    attention_maps = [[] for _ in range(num_layers)]  # saved for visualization in section 9
    all_logits = []   # "logits" = the raw output scores, before softmax. now you know the term.

    for pos in range(seq_len):

        tok_emb = weights["token_embed"][token_ids[pos]]
        pos_emb = weights["position_embed"][pos]

        combined = []
        for t, p in zip(tok_emb, pos_emb):
            combined.append(t + p)
        x = normalize(combined)

        for layer in range(num_layers):
            residual = x
            x = normalize(x)

            query = linear(x, weights[f"layer{layer}.query"])
            key = linear(x, weights[f"layer{layer}.key"])
            value = linear(x, weights[f"layer{layer}.value"])

            keys[layer].append(key)
            values[layer].append(value)

            scores = []
            for t in range(pos + 1):
                dot = Value(0)
                for j in range(embed_dim):
                    dot = dot + (query[j] * keys[layer][t][j])
                scores.append(dot * (embed_dim ** -0.5))


            attn_weights = softmax(scores)
            if return_attention:
                attention_maps[layer].append([w.data for w in attn_weights])
            attended = []
            for j in range(embed_dim):
                val_sum = Value(0)
                for t in range(pos + 1):
                    val_sum = val_sum + (attn_weights[t] * values[layer][t][j])
                attended.append(val_sum)


            x = linear(attended, weights[f"layer{layer}.attn_out"])
            x = [a + r for a, r in zip(x, residual)]

            ff_mid = linear(normalize(x), weights[f"layer{layer}.ff_up"])
            ff_hidden = [h.relu() for h in ff_mid]
            ff_out = linear(ff_hidden, weights[f"layer{layer}.ff_down"])
            x = [a + b for a, b in zip(ff_out, x)]

        all_logits.append(linear(x, weights["output_proj"]))
    if return_attention:
        return all_logits, attention_maps
    return all_logits

### You Be the Model

Before we define "loss," play one round of the model's game. What character comes next?

```
Peter Pi_
```

You said `p`, almost certainly, and you were confident. Maybe you'd bet 90% on `p` and spread the rest across a few other letters.

That bet IS the model's entire job: assign a probability to every possible next character. **Loss** is just the score of that bet. Confident and right: tiny loss. Confident and wrong: huge loss. Everything below is the bookkeeping for that idea.


---
## 6 · Loss Function

**Loss = "How wrong was the prediction?"**

Here's the intuition:
- Model predicts: `{'P': 10%, 'i': 30%, 'e': 5%, ...}`  
- Actual next char: `'i'`
- We look up the probability it assigned to `'i'`: **30%**
- Loss = `-log(0.30)` ≈ **1.2**

**Why `-log`?**  
- If the model was confident and correct (90% on right answer): loss ≈ 0.1 (good!)  
- If the model was wrong (1% on right answer): loss ≈ 4.6 (bad!)  
- `-log` turns "probability of correct answer" into "how much to punish"

**What to expect:**
- Random guessing among 24 chars = 1/24 ≈ 4% → loss ≈ 3.2
- Well-trained model might get loss down to 0.5-1.0

In [19]:
def compute_loss(token_sequence):
    """Average cross-entropy loss across all positions."""
    logits = gpt_forward(token_sequence[:-1])
    total_loss = Value(0)
    for pos, logit in enumerate(logits):
        probs = softmax(logit)
        total_loss = total_loss + (probs[token_sequence[pos + 1]].log() * -1)
    return total_loss * (1 / len(logits))

In [20]:
# Check initial loss — with random weights, expect loss ≈ ln(vocab_size)
# Why? Random model gives ~equal probability to all 24 chars: 1/24 ≈ 4%
# Loss = -log(1/24) = log(24) ≈ 3.18
expected_random_loss = math.log(vocab_size)
actual_initial_loss = compute_loss(val_data[:context_length + 1]).data

print(f"expected random loss: {expected_random_loss:.4f}")
print(f"actual initial loss:  {actual_initial_loss:.4f}")

expected random loss: 3.1355
actual initial loss:  3.3241


---
## 7 · Inference / Generation

**Generating text is just repeated prediction:**

1. Start with a prompt: `"Peter Piper "`
2. Feed it through the model → get probabilities for next character
3. **Sample** from those probabilities (not always pick the highest — adds variety)
4. Append the chosen character to the prompt
5. Repeat

**Before training:** The model's "brain" is random numbers, so it outputs gibberish.

**After training:** The same process produces recognizable text, because those numbers have been tuned to predict well.

**This loop is ChatGPT.** When ChatGPT streams an answer to you word by word, it is running exactly this loop: predict, sample, append, repeat. The typing effect is not theater; the model really is producing one token at a time, just like ours is about to.

In [21]:
def generate_text(prompt="Peter Piper ", num_chars=50, temperature=1.0):
    token_ids = []
    for c in prompt:
        token_ids.append(characters.index(c))

    print(prompt, end="", flush=True)

    for _ in range(num_chars):
        context = token_ids[-context_length:]
        logits = gpt_forward(context)
        last_logit = [l * (1 / temperature) for l in logits[-1]]  # temperature: <1 plays it safe, >1 gets adventurous
        probs = softmax(last_logit)

        weights_list = []
        for p in probs:
            weights_list.append(p.data)

        next_token = random.choices(range(vocab_size), weights=weights_list)[0]
        token_ids.append(next_token)
        print(characters[next_token], end="", flush=True)
    print()

In [22]:
generate_text()

Peter Piper 'iaeIe. eaowef'h,dc'ltfd '.awi.o, we,laPhdlrs hs?s


---
## 8 · Training Loop

**This is where learning happens.** 1,000 times, we:

![Training Loop](https://raw.githubusercontent.com/Infatoshi/yoctogpt/master/images/07-training-loop.png)

1. **Pick a random chunk** of training text (8 characters)
2. **Forward pass:** Predict each next character, measure total loss
3. **Backward pass:** Call `backward()` to get gradients for all 4,000 parameters
4. **Update:** Nudge each parameter slightly in the direction that reduces loss

```
for each of 1,000 steps:
    loss = "how wrong was I on this chunk?"
    loss.backward()           # compute all gradients
    for each parameter p:
        p = p - learning_rate * gradient   # nudge toward better
```

**The optimizer trick (RMSprop):** We don't just use the raw gradient. We divide by a running average of recent gradient magnitudes. This helps parameters that get big gradients take smaller steps (prevents overshooting).

**Watch the loss drop:** ~3.3 (random) → ~0.5 (trained). The model goes from 4% accuracy to roughly 60%+ on predicting the next character.

### What to watch for

As training runs, look for:

1. **Loss going down:** ~3.2 → ~1.2 → ~0.5
2. **Text improving:**
   - Step 0: Pure gibberish
   - Step 100: Word fragments (`"per"`, `"peck of"`)
   - Step 300-500: Real words chaining (`"pick peck of pick"`)
   - Step 1,000: Recognizable phrases (`"Peter Piper picked a..."`)

The model is literally learning English character patterns from scratch!

In [ ]:
num_steps = 1000
train_losses = []   # loss on each training chunk (noisy but honest)
val_history = []    # (step, avg val loss) every 100 steps

# average val loss over several windows: a single window is too noisy
# and makes the val number bounce around between checks
val_starts = range(0, len(val_data) - context_length, 4)

for step in range(num_steps + 1):
    if step % 100 == 0:
        val_loss = sum(compute_loss(val_data[i : i + context_length + 1]).data for i in val_starts) / len(val_starts)
        val_history.append((step, val_loss))
        print(f"\nstep {step}: val loss = {val_loss:.4f}")
        generate_text()

    start_idx = random.randint(0, len(train_data) - context_length - 1)
    chunk = train_data[start_idx : start_idx + context_length + 1]

    current_loss = compute_loss(chunk)
    train_losses.append(current_loss.data)
    current_loss.backward()

    # RMSprop-style update: divide each gradient by a running average of its
    # recent magnitude, so parameters with jumpy gradients take smaller steps
    for param in all_parameters:
        param.m = 0.99 * getattr(param, "m", 0.0) + 0.01 * (param.grad ** 2)
        param.data -= learning_rate * param.grad / (param.m ** 0.5 + 1e-8)
        param.grad = 0.0


### The Loss Curve

Numbers scrolling by are easy to ignore. A picture of the same numbers is undeniable.

**Why is the train line so jittery?** Because our *batch size* is 1: each step's gradient comes from a single 8-character window, so the signal is noisy. Real training averages gradients over thousands of windows at once (a "batch"), purely for smoothness and GPU efficiency: the model itself is unchanged. That's the entire concept of batching, and it's why we skip it here.

(matplotlib is used ONLY for plotting here; the model itself is still pure standard library)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(train_losses, alpha=0.3, label="train loss (per chunk)")
plt.plot([s for s, v in val_history], [v for s, v in val_history], "o-", label="val loss (avg of several windows)")
plt.axhline(math.log(vocab_size), linestyle="--", color="gray", label=f"random guessing = ln({vocab_size})")
plt.xlabel("training step")
plt.ylabel("loss")
plt.title("watch it learn: loss falling = predictions improving")
plt.legend()
plt.show()


---
## What You Just Witnessed

You trained a Transformer language model from scratch.

- **Step 0:** Random noise. `"Peter Piper 'sdd kPdl?,?"`
- **Step 1,000:** Recognizable patterns. `"Peter Piper picked a peck of..."`

The model learned that:
- `'P'` often follows `' '` (space)
- `"pick"` is a common substring
- `"peck"` and `"pepper"` are related patterns

All from adjusting 4,000 numbers based on "was I right about the next character?"

---

## What's the same in ChatGPT?

| This notebook | ChatGPT |
|---------------|---------|
| 4,000 parameters | trillions of parameters |
| 8-character context | hundreds of thousands of tokens |
| Character-level tokens | Subword tokens (~50k vocabulary) |
| 1 attention head | 96+ heads per layer |
| Tongue twister dataset | Huge chunk of the internet |
| 1 window per step (batch size 1) | Thousands of windows per step (batching) |
| Same core idea: predict next token | Same core idea |

The architecture is the same. The scale is different.

---
## 9 · Peek Inside the Brain: Attention Weights

Training is done, so let's look at what attention actually learned. For each token in `"Peter Pi"`, we plot how much it attends to every token before it.

How to read it:
- Each **row** is a token doing the predicting
- Bright cells in that row show which earlier tokens it found relevant
- The upper-right triangle is empty: tokens cannot see the future. That's causality, enforced by construction.


In [ ]:
prompt = "Peter Pi"
prompt_ids = [characters.index(c) for c in prompt]
_, attention_maps = gpt_forward(prompt_ids, return_attention=True)

attn = attention_maps[0]  # layer 0: row t holds t+1 weights (it can only see tokens 0..t)
grid = [[row[j] if j < len(row) else 0.0 for j in range(len(prompt_ids))] for row in attn]

plt.figure(figsize=(5, 5))
plt.imshow(grid, cmap="Blues")
plt.xticks(range(len(prompt_ids)), list(prompt))
plt.yticks(range(len(prompt_ids)), list(prompt))
plt.xlabel("attending to (earlier tokens)")
plt.ylabel("current token")
plt.title("learned attention weights")
plt.colorbar()
plt.show()


---
## 10 · Play With It

Different prompts, different randomness. Every run is a fresh sample from the model's learned probabilities.

(prompts can only use characters that appear in the training text)


In [ ]:
generate_text("A peck of ")
generate_text("If Peter ")
generate_text("pickled ")

A peck of pepick peper. ped f a picheped of er Pepick pisa p
If Peter of f ck pics rPers's. per peck w. Ped o, pepers pi
pickled per ler iled perr picketf ters pea s. ppa pickled 


In [ ]:
# TRY IT YOURSELF: temperature controls how adventurous sampling is
# temperature < 1: play it safe (picks the likely characters, more repetitive)
# temperature > 1: get weird (spreads probability out, more chaos)

generate_text("Peter Piper ", temperature=0.5)
generate_text("Peter Piper ", temperature=1.0)
generate_text("Peter Piper ", temperature=1.5)


---
## 11 · Train It on Your Own Text

The fastest way to make this stick: replace the dataset.

1. Scroll up to the first code cell and swap the tongue twister for your own text
2. `Runtime > Run all`
3. Watch the same ~4,000 numbers learn YOUR patterns

Tips:
- Keep it short and repetitive (200 to 400 characters). Tiny models love patterns.
- A song chorus, a poem, or your name and a fact about you repeated 3 times all work great.
- Prompts passed to `generate_text()` can only use characters that appear in your text.


---

## Next Steps

- **Go deeper:** [freeCodeCamp LLM course](https://www.youtube.com/watch?v=UU1WVnMk4E8) (7 hrs, uses PyTorch)
- **Original source:** [Karpathy's nanogpt lecture](https://www.youtube.com/watch?v=kCc8FmEb1nY) (2 hrs)
- **Make it fast:** [CUDA course](https://www.youtube.com/watch?v=86FAWCzIe_4) (12 hrs, GPU programming)
- **Understand backprop:** [Whiteboard explanation](https://youtu.be/0dbihoMRuyg?si=CvXDG6BC8khGcxoT)

---
## Appendix: Terminology Dictionary

Every scary ML term in this notebook, with the mechanics behind the word:

| Term | What it actually is |
|------|---------------------|
| **Token** | A piece of text swapped for a number, like an arcade token standing in for a coin (for us: one character) |
| **Embedding** | A token's point in 16-dimensional space; "embedded" from separate symbols into continuous numbers so *closeness* exists |
| **Position embedding** | A learned vector meaning "I'm in slot 3"; added because attention has no built-in sense of order |
| **Parameter / Weight** | One adjustable number in the brain (we have 3,968 of them) |
| **Projection / Linear layer** | Matrix × vector: one dot product per row, mapping a vector into a new space (16 numbers in, 24 out) |
| **Forward pass** | Running the model to make a prediction |
| **Loss** | The score of the model's bet: -log(probability it gave the right answer); lower = better |
| **Gradient** | For each parameter: which direction (and how strongly) to nudge it to reduce loss |
| **Backward pass** | Computing all 3,968 gradients via the chain rule |
| **Learning rate** | How big a nudge each step takes |
| **Attention** | A weighted average over earlier tokens, where the model *learns* the weights from content (Q·K scores) |
| **Normalization (RMS)** | Dividing a vector by √(mean of squares) so its typical size is ~1; stops values exploding or vanishing across layers |
| **Nonlinearity / ReLU** | max(0, x): the bend that stops stacked linear layers collapsing into one |
| **Softmax** | exp every score, divide by the sum: scores become probabilities that sum to 1 |
| **Logits** | The raw scores before softmax (one per character) |
| **KV cache** | Stored Key/Value vectors of earlier tokens so generation doesn't recompute them |

Don't memorize these; refer back as needed.
